# Spot the Differences

<img src="./img/org_differences.jpg"/>

- ArcGIS Enterprise and ArcGIS Online seem the same, but there are major differences in what you can do in terms of administration
- Enterprise has no concept of `credit` management
- ArcGIS Online has no concept of managing hardware or deep server configurations


# Managing Groups, Users and Content


- Let's start where the two overlap: groups, users, and content
- We saw how we can `search` for content, but let's start managing content

In [ ]:
from arcgis.gis import GIS, UserManager

In [ ]:
gis = GIS(profile='your_enterprise_profile', trust_env=True)

### The UserManager

- Allows for the creation, deletion and updating all things users
- You can:
    + reset passwords
    + add content to a user
    + update their information
    + understand license, user and other types
    + much, much more...

In [ ]:
um:UserManager = gis.users

#### Understanding Available Licensing

- When creating or managing user license, roles and other limited resources, you need to understand how many things you have and how many are left.
- The `counts` method returns a simple report of available license for roles, app, bundles and license types.
- Use these counts to guide automatic user creation and knowing when you have to review in-active accounts.

In [ ]:
for t in ['role', 'app', 'user_type']:
    print(f'Dispalying {t.upper()}')
    display(um.counts(t))

#### Examining Available License 

- This allows us to see what license types we can use to create a user

In [ ]:
lt:list[dict] = um.license_types
for t in lt:
    if t.get("state") =="active":
        print(t.get("id"),t.get('maxUsers'))

### Examining Roles

<center><img src="./img/rolls-picture.jpg" width=700/></center>

`roles` define the specific privileges (actions) a member can perform within the portal, such as creating content, publishing services, or administering the organization

 - You can have custom roles or use the built-in roles that comes with Enterprise and Online
    + Viewer
    + User
    + Publisher
    + Administrator

In [ ]:
from arcgis.gis import RoleManager, Role

In [ ]:
rm:RoleManager = um.roles

#### Examine All Custom Roles on the Site

In [ ]:
rm.all()

#### Get a Single Role with `get`

In [ ]:
role:Role = rm.get_role("Viewer")
role.privileges

#### Create a Custom Role

In [ ]:
custom_role = rm.create(name="DevSummitEditor", 
                 description="Allow to modify service data", 
                 privileges=[
                        "features:user:edit",
                        "features:user:fullEdit",
                        "opendata:user:designateGroup",
                        "portal:admin:viewUsers",
                        "portal:user:createGroup"]
                )
custom_role

### Creating a User

- Administrators can `create` a user using the `create` method

In [ ]:
import uuid
username = f"UCUser{uuid.uuid4().hex[:3]}"
password = f"!{uuid.uuid4().hex[:6]}A"

##### Call `create`

Using random data, we generate a new user using the custom role created above!

In [ ]:
user = um.create(username=username,
        password=password,
        firstname=uuid.uuid4().hex[:6],
        lastname=uuid.uuid4().hex[:6],
        email=uuid.uuid4().hex[:6] + "@esri.com",
        role=custom_role)
user

In [ ]:
user.role

#### Assign a New Role

- roles can be updated and changed for users

In [ ]:
user.update_role("org_publisher")

In [ ]:
user.role

In [ ]:
custom_role.delete()

#### Updating User Information

In [ ]:
user.update(first_name="Sammy", 
            last_name="GeoSpatial", 
            thumbnail=r"./img/user-icon.jpg")
user

#### Looking at Content

- using `folders` and `items` you can iterate through user's content

In [ ]:
for folder in user.folders:
    print(folder)

- Notice we have one folder, the root folder.  Every user will have this.

In [ ]:
for item in user.items():
    print(item)

- notice we have no items, the user was just created.

## Content and it's Management

<center><img src="./img/its-messy.jpg"/></center>

But it doesn't have to be.... 

Using `Folders` within the system help organize content.

In [ ]:
from arcgis.gis import ContentManager
cm:ContentManager = gis.content
folders = cm.folders

### Creating Folders

- Let's add a new folder to our newly created user

In [ ]:
folder = folders.create(folder="critical_infrastructure", owner=user, exist_ok=True)
folder

**Note:**

- If I do not specify the `owner` it will look at my account, not the newly created user

### Getting a Folder

In [ ]:
folders.get("critical_infrastructure", owner=user)

### Viewing the Folders

In [ ]:
for fld in folders.list(owner=user):
    print(fld)

In [ ]:
#user.delete()

## Adding  and Publishing Content

<center><img src="./img/oh-boy.gif"/></center>

In [ ]:
folder

In [ ]:
from arcgis.gis import ItemTypeEnum, ItemProperties

### Working with CSV Data

- always use `analyze` method when publishing CSV data
- when adding content use the `ItemProperties` dataclass over dictionaries
- Use the `ItemTypeEnum` over strings.

In [ ]:
data = r"./data/csv/springsteen.csv"
item_props:ItemProperties = ItemProperties(title='Springsteen NJ Location', 
                                           item_type=ItemTypeEnum.CSV,
                                           )
add_job = folder.add(item_properties=item_props, file=data)

In [ ]:
item = add_job.result()
item.update(thumbnail=r"./img/borntorun.jpg")
item

In [ ]:
params = cm.analyze(item=item, file_type="csv").get("publishParameters")

In [ ]:
params['name'] = "springsteenloc"  # update it's name (if you want)

In [ ]:
pub_job = item.publish(publish_parameters=params, future=True)
pitem = pub_job.result()

In [ ]:
pitem

#### Examine the Results

In [ ]:
m = gis.map('Asbury Park, NJ')
m

In [ ]:
m.content.add(pitem)

#### Working with Tables

In [ ]:
table_publish_parameters:dict = cm.analyze(item=item, file_type="csv").get("publishParameters")

In [ ]:
table_publish_parameters['name'] = 'frsthsttble' # this needs to be updated
table_publish_parameters['locationType'] = "none" # this makes it a hosted table
table_publish_parameters['layerInfo']['name'] = 'frsthsttble' # this needs to be updated

In [ ]:
pub_tbl_job = item.publish(publish_parameters=table_publish_parameters, future=True)
hst_tbl_item = pub_tbl_job.result()

In [ ]:
hst_tbl_item

#### Adding Shapefiles

In [ ]:
shapefile_item_properties:ItemProperties = ItemProperties(
    title="New Jersey Lighthouse Location",
    item_type=ItemTypeEnum.SHAPEFILE,
    tags="lighthouse,safety,boating,shipping",
    description="New Jersey Lighthouse Locations",
    snippet="A datasets of New Jersey's lighthouses"
)

In [ ]:
shp_fp:str = r"./data/shapefile/Lighthouse_Structures.zip"

In [ ]:
add_job = folder.add(item_properties=shapefile_item_properties, file=shp_fp)
lighthouse_item = add_job.result()
lighthouse_item

#### Publish the Lighthouse Dataset

In [ ]:
publish_job = lighthouse_item.publish(publish_parameters={"name":"lighthouses",
                                            "description" : "NJ Lighthouse Locations"},
                        future=True)
publish_job

In [ ]:
published_lighthouses = publish_job.result()
published_lighthouses

#### Adding File Geodatabases

- when adding the FGDB, you need to zip the whole directory of the .gdb folder

In [ ]:
fgdb_fp:str = r"./data/fgdb/Municipal_Centroids.zip"
fgdb_item_properties:ItemProperties = ItemProperties(
    title="Municipal Centroids", # REQUIRED
    item_type=ItemTypeEnum.FILE_GEODATABASE, # REQUIRED
    tags="Towns,City,Centroids",
    description="Municipal Centroid Locations",
    snippet="Municipal Centroid Locations"
)

In [ ]:
add_job = folder.add(item_properties=fgdb_item_properties, file=fgdb_fp)
centroid_item = add_job.result()
centroid_item

In [ ]:
publish_job = centroid_item.publish(publish_parameters={"name":"njmuncentroid",
                                            "description" : "NJ Municipal Centroid Locations"},
                        future=True)
publish_job

In [ ]:
publish_job.result()

## Working with Groups

<center><img src="./img/welcome-team-memes-group-huddle.jpg"/></center>

- A group is a collection of items usually related to a specific area of interest
- Create groups as a way to organize and share your items
- Group owners can decide who can find, join, and contribute to these groups

In [ ]:
from arcgis.gis import Group, GroupManager

In [ ]:
gm:GroupManager = gis.groups

### Searching for Groups

- like users and items, groups can be discovered
- use the search to find existing groups.

In [ ]:
gm.search(query="Disaster Response")

### Creating a Group

- groups can be create and updated

In [ ]:
grp:Group = gm.create('DevSummit Group 2026', tags='NJ')
grp

In [ ]:
grp.update(snippet="This is my devsummit group", thumbnail=r"./img/group-image.jpg")
grp

### Managing Group Users

- Administrators can add and remove users in groups

In [ ]:
grp.add_users(gis.users.search("a*"))

In [ ]:
grp.get_members()

#### Sharing Items with Groups

In [ ]:
item_share_mgr = pitem.sharing.groups
item_share_mgr

In [ ]:
item_share_mgr.add(grp)

In [ ]:
item_share_mgr.list()

### Deleting the Group

In [ ]:
grp.delete()

This does not impact any content on the organization. 